In [ ]:
!pip -q install pandas numpy scikit-learn tensorflow==2.19.0

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf

In [ ]:
from google.colab import files


df = pd . read_csv("/content/windows10_dataset.csv")
print("Shape:", df.shape)
print("Target columns:", [c for c in ["label","type"] if c in df.columns])
df.head()

Shape: (35975, 127)
Target columns: ['label', 'type']


,ts,Processor_DPC_Rate,Processor_pct_ Idle_Time,Processor_pct_ C3_Time,Processor_pct_ Interrupt_Time,Processor_pct_ C2_Time,Processor_pct_ User_Time,Processor_pct_ C1_Time,Processor_pct_ Processor_Time,Processor_C1_ransitions_sec,...,LogicalDisk(_Total) Avg Disk Write Queue Length,LogicalDisk(_Total) Avg Disk Queue Length,LogicalDisk(_Total) pct_ Disk Read Time,LogicalDisk(_Total) Disk Write Bytes sec,LogicalDisk(_Total) Disk Transfers sec,LogicalDisk(_Total) Avg Disk Bytes Transfer,LogicalDisk(_Total) pct_ Disk Write Time,LogicalDisk(_Total) Avg Disk sec Transfer,label,type
0,1554206309,4,29.90817156,0,0.078240397,0,61.02750947,29.90817156,66.2001486,480.0949912,...,0.134876119,0.163098103,2.822198461,1924988.236,402.5827836,9100.481592,13.48761188,0.000405118,0,normal
1,1554206319,9,31.75168186,0,0.312520973,0,59.14459419,31.75168186,66.24773489,427.0412837,...,0.157564294,0.189279353,3.17150584,1497232.743,355.1343322,11153.55449,15.75642941,0.000532995,0,normal
2,1554206329,5,29.49516707,0,1.16822183,0,46.02794011,29.49516707,66.90038148,1159.426821,...,0.401773775,0.782471877,38.06981014,1854228.536,760.4317685,26994.26595,40.17737752,0.001028323,0,normal
3,1554206339,12,18.22437505,0,1.097191902,0,47.80621859,18.22437505,79.54520811,736.622011,...,0.259062331,0.768934004,50.98716725,15912390.29,699.4999538,53141.6638,25.90623311,0.001099429,0,normal
4,1554206349,12,14.86118688,0,1.562431019,0,41.8731513,14.86118688,82.73513724,896.1085985,...,0.103836186,1.081729712,97.7893526,23609299.7,1184.550425,40988.68444,10.38361856,0.000912738,0,normal


In [ ]:
assert "label" in df.columns, "缺少二分类列 label"
assert "type" in df.columns, "缺少多分类列 type"

# 可选：去掉时间戳，避免泄漏/无意义噪声
drop_cols = ["ts"] if "ts" in df.columns else []

# 只取数值特征（AutoIoT论文输入是“高维统计特征向量”）
X_df = df.drop(columns=["label","type"] + drop_cols, errors="ignore")
X_df = X_df.select_dtypes(include=[np.number]).copy()

# inf -> nan -> 填充
X_df = X_df.replace([np.inf, -np.inf], np.nan)
X_df = X_df.fillna(X_df.median(numeric_only=True))

X = X_df.values.astype("float32")

# 多分类：type
le_type = LabelEncoder()
y_type = le_type.fit_transform(df["type"].astype(str)).astype("int32")
n_type = len(le_type.classes_)

# 二分类：label (0/1)
y_bin = df["label"].astype(int).values.astype("int32")

print("X:", X.shape, "y_type classes:", n_type, "y_bin:", np.unique(y_bin))

X: (35975, 55) y_type classes: 8 y_bin: [0 1]


In [ ]:
X_train, X_test, y_type_train, y_type_test, y_bin_train, y_bin_test = train_test_split(
    X, y_type, y_bin, test_size=0.30, random_state=42, stratify=y_type
)

# 标准化（只用训练集fit）
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

# 设定标注比例（例如 8%：论文中常测小比例标注）
label_pct = 0.08
n_train = X_train.shape[0]
idx = np.random.RandomState(42).permutation(n_train)

n_labeled = int(n_train * label_pct)
labeled_idx = idx[:n_labeled]
unlabeled_idx = idx[n_labeled:]

X_L = X_train[labeled_idx]
y_type_L = y_type_train[labeled_idx]
y_bin_L  = y_bin_train[labeled_idx]

X_U = X_train[unlabeled_idx]

print("Train total:", n_train, "Labeled:", len(X_L), "Unlabeled:", len(X_U))

Train total: 25182 Labeled: 2014 Unlabeled: 23168


In [ ]:
class AutoIoTClassifier(tf.keras.Model):
    def __init__(self, n_features, n_type_classes):
        super().__init__()
        self.n_features = n_features

        # CNN backbone: treat feature vector as 1D "sequence" length = n_features
        self.conv1 = tf.keras.layers.Conv1D(64, 3, padding="same", activation="relu")
        self.conv2 = tf.keras.layers.Conv1D(64, 3, padding="same", activation="relu")
        self.pool1 = tf.keras.layers.MaxPool1D(2)

        self.conv3 = tf.keras.layers.Conv1D(128, 3, padding="same", activation="relu")
        self.conv4 = tf.keras.layers.Conv1D(128, 3, padding="same", activation="relu")
        self.pool2 = tf.keras.layers.MaxPool1D(2)

        self.conv5 = tf.keras.layers.Conv1D(256, 3, padding="same", activation="relu")
        self.conv6 = tf.keras.layers.Conv1D(256, 3, padding="same", activation="relu")
        self.gap   = tf.keras.layers.GlobalAveragePooling1D()  # avg-pool

        # embedding layer (low-dim)
        self.emb = tf.keras.layers.Dense(128, activation=None)

        # shared FC (hard parameter sharing for multi-task) :contentReference[oaicite:5]{index=5}
        self.fc_shared = tf.keras.layers.Dense(128, activation="relu")
        self.drop = tf.keras.layers.Dropout(0.3)

        # head-1: specific classes (use "type" as proxy)
        self.head_type = tf.keras.layers.Dense(n_type_classes, activation=None)

        # head-2: IoT/non-IoT (use label as proxy)
        self.head_bin  = tf.keras.layers.Dense(2, activation=None)

    def call(self, x, training=False):
        # x: [B, n_features] -> [B, n_features, 1]
        x = tf.expand_dims(x, axis=-1)

        x = self.conv1(x); x = self.conv2(x); x = self.pool1(x)
        x = self.conv3(x); x = self.conv4(x); x = self.pool2(x)
        x = self.conv5(x); x = self.conv6(x)
        x = self.gap(x)

        z = self.emb(x)  # embedding (used by CCLP)
        h = self.fc_shared(tf.nn.relu(z))
        h = self.drop(h, training=training)

        logits_type = self.head_type(h)
        logits_bin  = self.head_bin(h)
        return z, logits_type, logits_bin

model = AutoIoTClassifier(n_features=X_train.shape[1], n_type_classes=n_type)

In [ ]:
def row_softmax(sim, temperature=0.5):
    sim = sim / temperature
    sim = sim - tf.reduce_max(sim, axis=1, keepdims=True)
    return tf.nn.softmax(sim, axis=1)

def compute_H(Z, temperature=0.5):
    # Z: [N, d], L2 normalize then cosine sim
    Z = tf.math.l2_normalize(Z, axis=1)
    sim = tf.matmul(Z, Z, transpose_b=True)  # [N,N]
    H = row_softmax(sim, temperature=temperature)
    return H

def label_propagation_FU(H_UU, H_UL, YL_onehot, eps=1e-6):
    # FU = (I - H_UU)^-1 H_UL YL
    N_U = tf.shape(H_UU)[0]
    I = tf.eye(N_U, dtype=H_UU.dtype)
    A = I - H_UU
    A = A + eps * tf.eye(N_U, dtype=H_UU.dtype)
    B = tf.matmul(H_UL, YL_onehot)
    FU = tf.linalg.solve(A, B)
    return FU

def compute_T(F):
    # Tij = sum_c fic fjc / mc, mc=sum_i fic
    mc = tf.reduce_sum(F, axis=0, keepdims=True)  # [1,C]
    mc = tf.maximum(mc, 1e-8)
    # numerator: F @ F^T -> [N,N]
    FFt = tf.matmul(F, F, transpose_b=True)
    # need divide by mc per class inside sum; do via (F / mc) @ F^T
    F_div = F / mc
    T = tf.matmul(F_div, F, transpose_b=True)
    return T, FFt

def cclp_loss(Z, yL, n_classes, NL, S=2, w_temp=0.5):
    """
    Z: embedded features of combined batch [N, d], N=NL+NU
    yL: int labels for labeled samples length NL
    """
    N = tf.shape(Z)[0]
    H = compute_H(Z, temperature=w_temp)  # [N,N]

    # split H blocks: H = [[HLL, HLU],[HUL, HUU]] (note paper layout)
    H_LL = H[:NL, :NL]
    H_LU = H[:NL, NL:]
    H_UL = H[NL:, :NL]
    H_UU = H[NL:, NL:]

    # YL onehot
    YL_onehot = tf.one_hot(yL, depth=n_classes, dtype=tf.float32)  # [NL,C]

    # FU via Label Propagation
    FU = label_propagation_FU(H_UU, H_UL, YL_onehot)  # [NU,C]

    F = tf.concat([YL_onehot, FU], axis=0)  # [N,C]
    T, FFt = compute_T(F)                  # T:[N,N], FFt:[N,N]
    M = FFt                                 # Eq(9)

    # multi-step H(s) approx: Hs = (H ⊙ M)^(s-1) @ H
    HM = H * M
    Hs = H
    if S > 1:
        HM_pow = tf.linalg.matmul(HM, tf.eye(tf.shape(HM)[0]))  # start = HM
        for _ in range(S - 1):
            Hs = tf.matmul(HM_pow, H)
            HM_pow = tf.matmul(HM_pow, HM)

    # cross-entropy between T and Hs: sum -Tij log(Hsij)
    Hs = tf.clip_by_value(Hs, 1e-8, 1.0)
    loss = -tf.reduce_mean(T * tf.math.log(Hs))
    return loss

In [ ]:
# 超参数（论文中 w 取 3 经验更好，文中提到过） :contentReference[oaicite:9]{index=9}
W_CCLP = 3.0
BATCH_L = 128
BATCH_U = 256  # NU >= NL
EPOCHS = 20
S_STEPS = 2

opt = tf.keras.optimizers.Adam(1e-3)
ce_type = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
ce_bin  = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# 构造 tf.data
ds_L = tf.data.Dataset.from_tensor_slices((X_L, y_type_L, y_bin_L)).shuffle(4096).repeat().batch(BATCH_L)
ds_U = tf.data.Dataset.from_tensor_slices((X_U,)).shuffle(4096).repeat().batch(BATCH_U)

it_L = iter(ds_L)
it_U = iter(ds_U)

steps_per_epoch = max(1, len(X_L) // BATCH_L)

@tf.function
def train_step(xL, yT, yB, xU):
    # combine labeled & unlabeled
    x = tf.concat([xL, xU], axis=0)
    NL = tf.shape(xL)[0]

    with tf.GradientTape() as tape:
        Z, logits_type, logits_bin = model(x, training=True)

        # labeled losses only
        logits_type_L = logits_type[:NL]
        logits_bin_L  = logits_bin[:NL]

        loss1 = ce_type(yT, logits_type_L)   # specific classes
        loss2 = ce_bin(yB, logits_bin_L)     # IoT/non-IoT proxy

        # CCLP on combined batch
        loss_cclp = cclp_loss(Z, yT, n_classes=n_type, NL=NL, S=S_STEPS, w_temp=0.5)

        loss_total = loss1 + loss2 + W_CCLP * loss_cclp

    grads = tape.gradient(loss_total, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return loss_total, loss1, loss2, loss_cclp

for epoch in range(1, EPOCHS + 1):
    losses = []
    for _ in range(steps_per_epoch):
        xL, yT, yB = next(it_L)
        (xU,) = next(it_U)
        lt, l1, l2, lc = train_step(xL, yT, yB, xU)
        losses.append([lt.numpy(), l1.numpy(), l2.numpy(), lc.numpy()])

    m = np.mean(losses, axis=0)
    print(f"Epoch {epoch:02d} | total={m[0]:.4f}  loss1(type)={m[1]:.4f}  loss2(bin)={m[2]:.4f}  cclp={m[3]:.4f}")

Epoch 01 | total=2.1455  loss1(type)=1.4438  loss2(bin)=0.6491  cclp=0.0175
Epoch 02 | total=1.6054  loss1(type)=0.9875  loss2(bin)=0.5653  cclp=0.0176
Epoch 03 | total=1.4696  loss1(type)=0.9138  loss2(bin)=0.5029  cclp=0.0176
Epoch 04 | total=1.2721  loss1(type)=0.8133  loss2(bin)=0.4062  cclp=0.0175
Epoch 05 | total=1.1245  loss1(type)=0.7683  loss2(bin)=0.3041  cclp=0.0174
Epoch 06 | total=0.9531  loss1(type)=0.6676  loss2(bin)=0.2339  cclp=0.0172
Epoch 07 | total=0.8570  loss1(type)=0.6049  loss2(bin)=0.2009  cclp=0.0171
Epoch 08 | total=0.7126  loss1(type)=0.5064  loss2(bin)=0.1551  cclp=0.0170
Epoch 09 | total=0.5850  loss1(type)=0.4027  loss2(bin)=0.1311  cclp=0.0171
Epoch 10 | total=0.5197  loss1(type)=0.3435  loss2(bin)=0.1252  cclp=0.0170
Epoch 11 | total=0.4843  loss1(type)=0.3092  loss2(bin)=0.1239  cclp=0.0171
Epoch 12 | total=0.4293  loss1(type)=0.2631  loss2(bin)=0.1155  cclp=0.0169
Epoch 13 | total=0.3550  loss1(type)=0.1992  loss2(bin)=0.1052  cclp=0.0169
Epoch 14 | t

In [ ]:
def predict_logits(X_in, batch=1024):
    Zs, lt, lb = [], [], []
    for i in range(0, len(X_in), batch):
        x = tf.convert_to_tensor(X_in[i:i+batch], dtype=tf.float32)
        z, a, b = model(x, training=False)
        lt.append(a.numpy()); lb.append(b.numpy())
    return np.concatenate(lt, axis=0), np.concatenate(lb, axis=0)

logits_type_test, logits_bin_test = predict_logits(X_test)

pred_type = np.argmax(logits_type_test, axis=1)
pred_bin  = np.argmax(logits_bin_test, axis=1)

print("=== Task A (proxy of 'specific IoT classes'): type multi-class ===")
print("Accuracy:", accuracy_score(y_type_test, pred_type))
print(classification_report(y_type_test, pred_type, target_names=le_type.classes_, digits=4))

print("\n=== Task B (proxy of 'IoT / non-IoT'): label binary ===")
print("Accuracy:", accuracy_score(y_bin_test, pred_bin))
print(classification_report(y_bin_test, pred_bin, digits=4))

=== Task A (proxy of 'specific IoT classes'): type multi-class ===
Accuracy: 0.9432039284721578
              precision    recall  f1-score   support

        ddos     0.9391    0.9928    0.9652      1382
         dos     0.6209    0.6013    0.6109       158
   injection     0.4967    0.8261    0.6204       184
        mitm     0.0000    0.0000    0.0000         4
      normal     0.9723    0.9591    0.9657      7462
    password     0.9942    0.9430    0.9679      1088
    scanning     0.0000    0.0000    0.0000       134
         xss     0.7875    0.9921    0.8780       381

    accuracy                         0.9432     10793
   macro avg     0.6013    0.6643    0.6260     10793
weighted avg     0.9380    0.9432    0.9393     10793


=== Task B (proxy of 'IoT / non-IoT'): label binary ===
Accuracy: 0.9533957194477902
              precision    recall  f1-score   support

           0     0.9808    0.9512    0.9658      7462
           1     0.8976    0.9583    0.9270      3331

   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
